<a href="https://colab.research.google.com/github/Lolla-data-analyst/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%pip install -q duckdb huggingface_hub

In [4]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

print("Connected successfully")

Connected successfully


In [5]:
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

In [7]:
rel = "hf://datasets/FlyRank/internship-warehouse"

test = con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 5")
test.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [9]:
result = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
GROUP BY report_date, client_hash_id, content_hash_id
ORDER BY row_count DESC
LIMIT 10
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬───────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ row_count │
│    date     │         varchar         │         varchar          │   int64   │
├─────────────┼─────────────────────────┼──────────────────────────┼───────────┤
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_39d7361b4945d504 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_cec711b02f3bbde6 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_275b6f7f733016d4 │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_ceaec531566ffcfc │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_755d951187fcd70a │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_7a7d3c7aa7cdfc5c │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_92c381fbd361212e │         1 │
│ 2026-03-01  │ client_62f4a7e64f5e0096 │ content_97188a7032a705cf │         1 │
│ 2026-03-01  │ client_62f4a

In [11]:
result = con.sql(f"""
SELECT MIN(report_date), MAX(report_date), Count(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' AND gsc_data_available = true
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────────────┬───────────┐
│ min(report_date) │ max(report_date) │ row_count │
│       date       │       date       │   int64   │
├──────────────────┼──────────────────┼───────────┤
│ 2026-03-01       │ 2026-03-31       │   3611061 │
└──────────────────┴──────────────────┴───────────┘



In [12]:
result = con.sql(f"""
SELECT COUNT(*) AS total_rows,
COUNT(CASE WHEN gsc_data_available = true THEN 1 ELSE NULL END) AS available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
result.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┐
│ total_rows │ available_rows │
│   int64    │     int64      │
├────────────┼────────────────┤
│    9841378 │        3611061 │
└────────────┴────────────────┘

